In [1]:
# 第一部分：导入必要的库
# ============================================

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')


In [2]:
# 第二部分：可调参数配置区（PCA + 线性SVM 经典对照组）
# ============================================

# 数据相关参数（与量子实验保持一致）
SAMPLE_RATIO = 1            # 数据采样比例
TEST_SIZE = 0.2             # 测试集比例
PCA_COMPONENTS = 8          # PCA降维维度（与量子比特数一致）

# 降维方法
DIM_REDUCTION_METHOD = 'pca'

# SVM 参数
SVM_KERNEL = 'linear'       # 核函数
SVM_C = 1.0                 # 正则化参数（默认值，不调优）

# 随机种子（与量子实验一致）
SEED = 42

# 初始化设置
# ============================================
np.random.seed(SEED)


In [3]:
# 第三部分：数据加载和处理函数（PCA降维，与量子实验完全一致）
# ============================================

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

def load_mnist_data():
    """加载MNIST数据集（读取本地 mnist.npz，无需 torchvision）"""
    import os

    data_path = "./mnist.npz"
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"未找到本地数据文件 {data_path}，请将其放在 notebook 同级目录。"
        )

    print(f"正在加载本地MNIST数据集: {data_path}")
    data = np.load(data_path)

    # mnist.npz 中数值已归一化到 [0,1]，形状为 (N,28,28)
    train_images = data['tr_x'].astype(np.float32).reshape(len(data['tr_x']), -1)
    train_labels = data['tr_y'].astype(np.int64)
    test_images = data['te_x'].astype(np.float32).reshape(len(data['te_x']), -1)
    test_labels = data['te_y'].astype(np.int64)

    # 合并数据并采样
    all_images = np.concatenate([train_images, test_images])
    all_labels = np.concatenate([train_labels, test_labels])
    sampled_indices = np.random.choice(
        len(all_images),
        int(len(all_images) * SAMPLE_RATIO),
        replace=False
    )
    sampled_images = all_images[sampled_indices]
    sampled_labels = all_labels[sampled_indices]

    print(f"总样本数: {len(all_images)}, 采样后: {len(sampled_images)}")
    print(f"类别分布: {np.unique(sampled_labels, return_counts=True)}")

    return sampled_images, sampled_labels

def preprocess_data(images, labels, pca=None, scaler=None, is_train=True):
    """
    数据预处理：PCA降维 + MinMax归一化到[-1,1]（与量子实验一致）
    训练集 fit，测试集 transform；返回 (features, pca, scaler)
    """
    if is_train:
        pca = PCA(n_components=PCA_COMPONENTS)
        pca_data = pca.fit_transform(images)
        print(f"PCA解释方差比: {np.sum(pca.explained_variance_ratio_):.4f}")

        scaler = MinMaxScaler(feature_range=(-1, 1))
        scaled_data = scaler.fit_transform(pca_data)
    else:
        pca_data = pca.transform(images)
        scaled_data = scaler.transform(pca_data)

    return scaled_data, pca, scaler


In [4]:
# 第四部分：SVM 训练与评估函数
# ============================================

def train_evaluate_svm(train_features, train_labels, test_features, test_labels,
                       kernel='linear', C=1.0):
    """训练SVM，返回 (模型, 训练集准确率, 测试集准确率)"""
    print(f"\n训练 SVM (kernel={kernel}, C={C}) ...")
    model = SVC(kernel=kernel, C=C, random_state=SEED)
    model.fit(train_features, train_labels)

    train_pred = model.predict(train_features)
    test_pred = model.predict(test_features)

    train_acc = (train_pred == train_labels).mean()
    test_acc = (test_pred == test_labels).mean()

    print(f"  训练集准确率: {train_acc:.4f}")
    print(f"  测试集准确率: {test_acc:.4f}")

    return model, train_acc, test_acc


In [5]:
# 第五部分：主程序 - PCA + 线性SVM 经典对照组
# ============================================

if __name__ == "__main__":
    print("=" * 60)
    print("PCA降维 + 线性SVM 经典对照组")
    print("=" * 60)

    # 1. 加载数据
    print("\n步骤1：加载MNIST数据")
    all_images, all_labels = load_mnist_data()
    print(f"数据大小: {len(all_images)}")

    # 2. 划分数据集（与量子实验一致）
    print("\n步骤2：划分训练集和测试集")
    train_images, test_images, train_labels, test_labels = train_test_split(
        all_images, all_labels, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
    )
    print(f"训练集: {len(train_images)} 样本, 测试集: {len(test_images)} 样本")

    # 3. PCA降维 + 归一化
    print("\n步骤3：PCA降维与归一化")
    train_features, pca, scaler = preprocess_data(train_images, train_labels, is_train=True)
    test_features, _, _ = preprocess_data(test_images, test_labels, pca=pca, scaler=scaler, is_train=False)
    print(f"特征形状 - 训练集: {train_features.shape}, 测试集: {test_features.shape}")

    # 4. 训练并评估 SVM
    print("\n步骤4：训练并评估线性SVM")
    model, train_acc, test_acc = train_evaluate_svm(
        train_features, train_labels, test_features, test_labels,
        kernel=SVM_KERNEL, C=SVM_C
    )

    # 5. 结果汇总
    print("\n" + "=" * 60)
    print("PCA + 线性SVM 对照组结果汇总")
    print("=" * 60)
    print(f"核函数: {SVM_KERNEL}, C: {SVM_C}")
    print(f"训练集准确率: {train_acc:.4f}")
    print(f"测试集准确率: {test_acc:.4f}")

    # 6. 保存结果
    np.savez('pca_svm_control_results.npz',
             kernel=SVM_KERNEL,
             C=SVM_C,
             train_acc=train_acc,
             test_acc=test_acc)
    print("\n结果已保存到 pca_svm_control_results.npz")


PCA降维 + 线性SVM 经典对照组

步骤1：加载MNIST数据
正在加载本地MNIST数据集: ./mnist.npz
总样本数: 70000, 采样后: 70000
类别分布: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int64), array([6903, 7877, 6990, 7141, 6824, 6313, 6876, 7293, 6825, 6958],
      dtype=int64))
数据大小: 70000

步骤2：划分训练集和测试集
训练集: 56000 样本, 测试集: 14000 样本

步骤3：PCA降维与归一化
PCA解释方差比: 0.4384
特征形状 - 训练集: (56000, 8), 测试集: (14000, 8)

步骤4：训练并评估线性SVM

训练 SVM (kernel=linear, C=1.0) ...
  训练集准确率: 0.8185
  测试集准确率: 0.8174

PCA + 线性SVM 对照组结果汇总
核函数: linear, C: 1.0
训练集准确率: 0.8185
测试集准确率: 0.8174

结果已保存到 pca_svm_control_results.npz
